<a href="https://colab.research.google.com/github/kupalmananalsal-hub/Project_Pi/blob/main/notebooks/train_thermal_human_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project Pi Thermal Human Detection Training

Run this notebook in Colab or another training machine. It downloads/prepares the MLX90640-compatible datasets, trains a small CNN, and exports `thermal_human_detector.tflite`.

In [1]:
!git clone https://github.com/kupalmananalsal-hub/Project_Pi.git || true
%cd Project_Pi
!pip install -q numpy pillow h5py tensorflow kaggle matplotlib

Cloning into 'Project_Pi'...
remote: Enumerating objects: 765, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 765 (delta 2), reused 2 (delta 2), pack-reused 759 (from 1)
Receiving objects: 100% (765/765), 329.06 KiB | 2.32 MiB/s, done.
Resolving deltas: 100% (406/406), done.
/content/Project_Pi


If you want to include the Kaggle YOLO thermal dataset, upload `kaggle.json` to Colab and run the Kaggle credential cell before downloading.

In [2]:
# Optional Kaggle credentials
# from google.colab import files
# files.upload()  # upload kaggle.json
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/kaggle.json
# !chmod 600 ~/.kaggle/kaggle.json

In [3]:
!python raspberry_pi/thermal/dataset_downloader.py --datasets thermo_presence mldetection skku_thermal_human yolov8_thermal


==> thermo_presence
Cloning into '/root/thesis_dataset/thermal/raw/thermo_presence'...
remote: Enumerating objects: 131, done.
remote: Counting objects: 100% (131/131), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 131 (delta 65), reused 79 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (131/131), 40.52 MiB | 21.19 MiB/s, done.
Resolving deltas: 100% (65/65), done.

==> mldetection
Downloaded /root/thesis_dataset/thermal/archives/mldetection_dataset.tar.gz
/content/Project_Pi/raspberry_pi/thermal/dataset_downloader.py:144: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  archive.extractall(destination)

==> skku_thermal_human
Cloning into '/root/thesis_dataset/thermal/raw/skku_thermal_human'...
remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (8/8), done.
re

In [4]:
!python raspberry_pi/thermal/preprocess_thermal_datasets.py --dataset all

INFO:thermal_preprocess:thermo_presence: accepted samples=0
INFO:thermal_preprocess:mldetection: accepted samples=0
INFO:thermal_preprocess:skku: accepted samples=0
INFO:thermal_preprocess:yolo: accepted samples=15486
INFO:thermal_preprocess:recorded: accepted samples=0
INFO:thermal_preprocess:Wrote 15486 samples to /root/thesis_dataset/thermal/processed/thermal_human_detection.npz
INFO:thermal_preprocess:Wrote preprocessing report to /root/thesis_dataset/thermal/processed/thermal_human_detection.report.json


In [13]:
%cd /content/Project_Pi
!pwd
!ls raspberry_pi/thermal/train_thermal_model.py

/content/Project_Pi
/content/Project_Pi
raspberry_pi/thermal/train_thermal_model.py


In [11]:
!find /content -type f \( -name "thermal_human_detector*" -o -name "split.json" \) -print

In [12]:
from pathlib import Path
import numpy as np

data_path = Path.home() / "thesis_dataset/thermal/processed/thermal_human_detection.npz"
print("Data exists:", data_path.exists())
print("Path:", data_path)

if data_path.exists():
    data = np.load(data_path, allow_pickle=True)
    print("Keys:", data.files)
    for key in ["frames", "masks", "presence", "source_ids", "input_domains"]:
        if key in data:
            print(key, data[key].shape)

Data exists: True
Path: /root/thesis_dataset/thermal/processed/thermal_human_detection.npz
Keys: ['frames', 'masks', 'presence', 'coverage', 'sources', 'source_ids', 'source_units', 'input_domains', 'dataset_names', 'annotation_types']
frames (15486, 24, 32)
masks (15486, 24, 32)
presence (15486,)
source_ids (15486,)
input_domains (15486,)


In [19]:
!python raspberry_pi/thermal/train_thermal_model.py \
  --data /root/thesis_dataset/thermal/processed/thermal_human_detection.npz \
  --output-dir raspberry_pi/thermal/models \
  --epochs 3 \
  --batch-size 64 \
  --split-by source_id

2026-05-25 17:32:28.411305: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-25 17:32:34.649410: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1779730354.650995    7897 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
Model: "functional"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━

In [20]:
!ls -lh raspberry_pi/thermal/models/

total 3.4M
-rw-r--r-- 1 root root 3.2M May 25 17:33 split.json
-rw-r--r-- 1 root root 169K May 25 17:33 thermal_human_detector.keras
-rw-r--r-- 1 root root 1.9K May 25 17:33 thermal_human_detector.metadata.json
-rw-r--r-- 1 root root 6.8K May 25 17:33 thermal_human_detector_metrics.json
-rw-r--r-- 1 root root  673 May 25 17:33 thermal_human_detector_pr_curve.csv
-rw-r--r-- 1 root root  18K May 25 17:33 thermal_human_detector_pr_curve.png
-rw-r--r-- 1 root root  27K May 25 17:33 thermal_human_detector.tflite


In [21]:
!python raspberry_pi/thermal/train_thermal_model.py \
  --data /root/thesis_dataset/thermal/processed/thermal_human_detection.npz \
  --output-dir raspberry_pi/thermal/models \
  --epochs 40 \
  --batch-size 64 \
  --split-by source_id

2026-05-25 17:34:13.924105: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-25 17:34:19.434270: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1779730459.435767    9115 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
Model: "functional"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━

In [22]:
from google.colab import files

for path in [
    "raspberry_pi/thermal/models/thermal_human_detector.tflite",
    "raspberry_pi/thermal/models/thermal_human_detector.metadata.json",
    "raspberry_pi/thermal/models/thermal_human_detector_metrics.json",
    "raspberry_pi/thermal/models/thermal_human_detector_pr_curve.png",
    "raspberry_pi/thermal/models/split.json",
]:
    files.download(path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>